In [1]:
import random, math
from collections import defaultdict
corpus = [
 "review notes for 10 minutes before class <eos>",
 "write three practice questions after each lesson <eos>",
 "take a 5 minute break after focused study <eos>",
 "solve two examples before checking the answer <eos>",
 "summarize one chapter in your own words <eos>",
]
def gold_reward(sentence):
    """1 iff actionable verb + numeric minutes + at most 12 words; otherwise 0."""
    words=[w.lower().strip(".,!?") for w in sentence.split() if w!="<eos>"]
    actions={"review","write","take","solve","summarize","practice"}
    return float(any(w in actions for w in words) and any(w.isdigit() for w in words)
                 and "minutes" in words and len(words)<=12)
follow=defaultdict(list)
for sentence in corpus:
    words=["<bos>"]+sentence.split()
    for a,b in zip(words,words[1:]): follow[a].append(b)
def sample_reference(rng,max_words=12):
    word,out="<bos>",[]
    while word!="<eos>" and len(out)<max_words:
        options=follow.get(word,[])
        if not options: break
        word=rng.choice(options); out.append(word)
    return " ".join(out)
print("Corpus size:",len(corpus)," Example reward:",gold_reward(corpus[0]))


Corpus size: 5  Example reward: 1.0


In [2]:
rng=random.Random(0)
baseline_samples=[sample_reference(rng) for _ in range(200)]
baseline_rewards=[gold_reward(s) for s in baseline_samples]
for s in baseline_samples[:6]: print(f"{gold_reward(s):.0f}  {s}")
print(f"Reference mean reward: {sum(baseline_rewards)/len(baseline_rewards):.3f}")


0  solve two examples before checking the answer <eos>
0  take a 5 minute break after focused study <eos>
1  review notes for 10 minutes before checking the answer <eos>
0  solve two examples before class <eos>
1  review notes for 10 minutes before class <eos>
0  write three practice questions after focused study <eos>
Reference mean reward: 0.230


Reference sampler baseline
This follows observed word transitions. It is not RL: reward is printed for evaluation but does not update sampler parameters.

In [3]:
candidates=[
 "review notes for 10 minutes before class",
 "write three practice questions after each lesson",
 "take a 5 minute break after focused study",
 "solve two examples before checking the answer",
 "summarize one chapter in your own words",
 "read the chapter and enjoy the topic",
 "organize your desk before beginning",
 "practice recall for 15 minutes tonight",
]
def softmax(logits):
    e=[math.exp(x-max(logits)) for x in logits]; z=sum(e)
    return [v/z for v in e]
def train(episodes=3000,lr=0.08,seed=7):
    rng=random.Random(seed); logits=[0.0]*len(candidates); avg=0.0
    for t in range(1,episodes+1):
        probs=softmax(logits)
        a=rng.choices(range(len(candidates)),weights=probs,k=1)[0]
        reward=gold_reward(candidates[a]); avg+=(reward-avg)/t
        for i,p in enumerate(probs):
            logits[i]+=lr*(reward-avg)*((1.0 if i==a else 0.0)-p)
    return logits
logits=train(); policy_probs=softmax(logits)
for s,p in zip(candidates,policy_probs): print(f"{p:.3f} reward={gold_reward(s):.0f}  {s}")
policy_mean=sum(p*gold_reward(s) for s,p in zip(candidates,policy_probs))
baseline_mean=sum(baseline_rewards)/len(baseline_rewards)
print(f"Policy expected reward: {policy_mean:.3f}")
print(f"Reference mean reward: {baseline_mean:.3f}")
print(f"Difference: {policy_mean-baseline_mean:+.3f}")


0.990 reward=1  review notes for 10 minutes before class
0.001 reward=0  write three practice questions after each lesson
0.001 reward=0  take a 5 minute break after focused study
0.001 reward=0  solve two examples before checking the answer
0.001 reward=0  summarize one chapter in your own words
0.001 reward=0  read the chapter and enjoy the topic
0.001 reward=0  organize your desk before beginning
0.006 reward=1  practice recall for 15 minutes tonight
Policy expected reward: 0.996
Reference mean reward: 0.230
Difference: +0.766


Interpretation and limitations
The policy learns to prefer entries in the menu that satisfy the rule; it does not compose new tips. The hand-written deterministic reward measures style, not educational effectiveness. The Markov sampler is not a neural language model. No state transition or delayed reward is modeled. A larger project could use learner progress as state and later quiz performance as reward.
Run in Colab
Upload this notebook using File → Upload notebook, then choose Runtime → Run all. No external packages, API keys, or GPU are needed. Save the executed notebook with outputs for GitHub.

Train a contextual-bandit policy
The action space is eight fixed tips. A softmax gives action probabilities. REINFORCE updates each logit by learning_rate × (reward − running_mean) × (indicator for selected action − action probability). This is a one-step policy-gradient experiment.

RL for Concise Study Tips
A small contextual-bandit experiment with a corpus baseline.
This Colab project replaces the starter weather corpus with campus study tips. It compares the corpus-based next-word sampler with a policy that learns which complete tip to choose. The trainable experiment is a one-step contextual bandit: each action selects a candidate tip and receives its deterministic style reward. It is not multi-step RL or a general language model.